# Chapter 29. Weak Ricci curvature bounds I: Definition and Stability

_Source span: printed pp. 795-846; physical PDF pp. 817-868._

This notebook is a standalone computational lesson for the unit named above. The local PDF is used as a map of topics and order, but the explanations, examples, code, and visuals here are rebuilt from scratch. The guiding question is: **Present weak Ricci bounds as convexity of entropy-like functionals along Wasserstein geodesics, then test stability.**

The chapter is treated as a working laboratory rather than a summary. We will translate the mathematical vocabulary into arrays, couplings, graphs, curves, density profiles, and checks that can fail if the idea has been misunderstood. That is especially important in optimal transport, where a word such as "plan", "map", "geodesic", or "curvature" has both a formal definition and a visible computational footprint. The notebook keeps those two readings next to each other.


In [ ]:
from pathlib import Path
import json
import sys
import numpy as np

try:
    HERE = Path(__file__).resolve().parent
except NameError:
    HERE = Path.cwd()

BOOK_ROOT = HERE
while BOOK_ROOT.name != "Optimal-Transport-Old-and-New":
    if BOOK_ROOT.parent == BOOK_ROOT:
        raise RuntimeError("Could not locate course root")
    BOOK_ROOT = BOOK_ROOT.parent

for candidate in [BOOK_ROOT, BOOK_ROOT / "scripts"]:
    if str(candidate) not in sys.path:
        sys.path.insert(0, str(candidate))

from scripts.otonn_inventory import UNIT_BY_ID
from utils.artifacts import display_artifact
from utils.transport import cost_matrix, demo_measures, exact_plan, plan_checks, sinkhorn_demo_value
from utils.validation import assert_artifacts, assert_check_flag
from utils.visuals import artifact_paths_for_unit, build_unit_artifacts

UNIT_ID = "chapter-29"
EXPECTED_MODE = "weak-ricci"
unit = UNIT_BY_ID[UNIT_ID]
assert unit["mode"] == EXPECTED_MODE


## Translation Guide

This unit's source terms are: `weak Ricci, CD(K,N), stability, entropy convexity`. In the notebook they become concrete objects:

- CD(K,N) style definitions are synthetic because they use measure geodesics.
- Distortion coefficients encode the chosen curvature and dimension.
- The definition is designed to survive convergence of metric-measure spaces.

The computational route is deliberately small. We start with a discrete transport problem because it exposes every marginal, cost entry, and unit of moved mass. We then use the unit's specialized visual mode, `weak-ricci`, to focus on the chapter's own geometry. Finally, a sanity cell checks that the generated artifact exists and that the numerical invariant attached to the unit actually passed.

That rhythm is meant to model how to read the book actively. A theorem about existence, a statement about Ricci curvature, or a definition of displacement convexity should leave behind something inspectable: a matrix with the right sums, a path whose mass remains one, a graph whose implications are connected, or a curve whose inequality is numerically visible.


In [ ]:
source_inventory = {
    "unit": unit["id"],
    "label": unit["label"],
    "title": unit["title"],
    "printed_pages": unit["printed"],
    "pdf_pages": unit["pdf"],
    "visual_mode": unit["mode"],
    "terms": unit["terms"],
}
source_inventory


## Concept Lens

1. CD(K,N) style definitions are synthetic because they use measure geodesics. 2. Distortion coefficients encode the chosen curvature and dimension. 3. The definition is designed to survive convergence of metric-measure spaces.

The common optimal-transport skeleton underneath this unit is the Monge-Kantorovich relaxation. We choose a source measure, a target measure, and a cost matrix. POT solves the finite linear program and returns a plan whose rows and columns must be the prescribed marginals. This modest computation is not a replacement for the theorem; it is a microscope for the theorem's bookkeeping. If the chapter is about duality, the active entries should align with zero slack. If it is about Wasserstein distance, the same matrix becomes a metric computation. If it is about displacement interpolation, the nonzero entries become moving weighted particles. If it is about Ricci curvature, the same interpolation language becomes a place to measure convexity and volume distortion.

The unit-specific layer is: **Entropy convexity curves with distortion-coefficient correction terms.** This representation was chosen because it makes the chapter's invisible condition visible. The learner should keep asking which object is fixed, which object is optimized, and which invariant is being checked. For this unit, the attached invariant is: **The corrected convexity residual is nonnegative for the displayed stable family.**


In [ ]:
artifact_result = build_unit_artifacts(unit)
paths = artifact_paths_for_unit(unit)
artifact_result, paths


In [ ]:
display_artifact(paths["figure"], width=820)
display_artifact(paths["checks"], width=820)


## Visual Reading

What to inspect: Entropy convexity curves with distortion-coefficient correction terms. The figure is not meant as decoration; it is a compact diagnostic for the chapter's main move. First locate the conserved quantity or structural relation. In a coupling diagram, that means row and column sums. In a graph, it means reachability of the conceptual route. In curvature and convexity plots, it means the ordering or residual required by the inequality. In a field or heatmap, it means the sign, jump, or determinant that the prose claims should be there.

The adjacent JSON check is part of the lesson. It records the invariant in machine-readable form so that the visual claim is not just a picture. A good habit is to read the figure, predict which Boolean check should be true, then open the JSON and see whether the computation agrees.


In [ ]:
source, target = demo_measures(UNIT_ID)
C = cost_matrix(source.points, target.points, p=2)
plan = exact_plan(source.weights, target.weights, C)
transport_summary = plan_checks(plan, source, target, p=2)
transport_summary


In [ ]:
mode_probe = {
    "mode": EXPECTED_MODE,
    "nonzero_plan_entries": int(np.count_nonzero(plan > 1e-10)),
    "largest_moved_mass": float(plan.max()),
    "source_weight_total": float(source.weights.sum()),
    "target_weight_total": float(target.weights.sum()),
}
if UNIT_ID == "chapter-06":
    mode_probe["geomloss_sinkhorn_probe"] = sinkhorn_demo_value()
mode_probe


## Lab: Make the Theorem Negotiable

Vary K and locate the threshold where the toy family no longer passes the residual check.

The point of the lab is not to produce a polished theorem. It is to make one hypothesis move while the rest of the setup stays fixed. In optimal transport this is often the quickest way to see why a theorem is phrased with care. A small change in a cost entry can move support from one edge to another. A small change in a density cap can make an interpolation violate a bound. A small change in curvature can reverse the order of a distortion curve.

For this unit, start from the generated data above. Keep the source and target arrays visible, change only one ingredient, and rerun the plan or artifact cell. Record whether the invariant still passes. If it fails, identify whether the failure is a mathematical obstruction, a numerical resolution issue, or simply a sign that the toy model no longer represents the chapter's assumptions.


## Takeaways

- Source span used: printed pp. 795-846; PDF pp. 817-868.
- The chapter vocabulary is rebuilt around `weak Ricci, CD(K,N), stability, entropy convexity`.
- The durable artifact lives under `artifacts/chapter-29-weak-ricci-curvature-bounds-i-definition-and-stability` and records both a figure and a JSON invariant check.
- POT supplies the finite Monge-Kantorovich plan used by the common transport lab. GeomLoss is smoke-tested in the course stack and probed in the Wasserstein-distance unit.
- The final sanity cell below checks artifact size, source-map consistency, mass conservation, and the unit-specific invariant.


In [ ]:
final_sanity = {
    "unit_id": UNIT_ID,
    "source_span": "printed pp. 795-846; PDF pp. 817-868",
    "transport_mass_conserved": bool(transport_summary["mass_conserved"]),
    "artifact_paths": {name: str(path.relative_to(BOOK_ROOT)) for name, path in paths.items()},
}
assert unit["printed"] == "795-846"
assert unit["pdf"] == "817-868"
assert transport_summary["mass_conserved"]
assert_artifacts(paths.values())
check_data = assert_check_flag(paths["checks"])
final_sanity["artifact_invariant_ok"] = bool(check_data["invariant_ok"])
final_sanity
